# Paper 4 – Data collection for the Kerala 2018 and 2019 floods

This notebook produces three inputs for the manuscript (Section 5, Table 2):

| Part | Output | Used in the paper as |
|---|---|---|
| A. Sentinel-1 flood extent | flood mask per acquisition date (GeoTIFF) + flooded area per district | observed flood extent, realized demand |
| B. WorldPop population | exposed and total population per district and date | $P_{i,t}$, $P_i$ |
| C. OpenStreetMap roads | drivable road graph, district-to-district travel times, flooded road segments | $\tau^0_{j,i}$, road closures |

**Before you start**
1. A Google Earth Engine account linked to a Google Cloud project (https://code.earthengine.google.com/register). Put the project ID in the cell below.
2. Runtime → Run all. Earth Engine exports go to a Google Drive folder and take roughly 10–40 minutes; the notebook waits for them.

All outputs go to `My Drive/Paper4_Kerala/`.

In [ ]:
!pip -q install earthengine-api geemap osmnx rasterio geopandas

In [ ]:
# ===================== SETTINGS (edit these) =====================
EE_PROJECT   = 'your-earth-engine-project-id'   # <-- change
DRIVE_FOLDER = 'Paper4_Kerala'

# Pre-flood reference (dry, pre-monsoon) and flood windows for each event.
# Every Sentinel-1 date inside the flood window is processed separately,
# so you get one flood map per acquisition (= a time series for the hindcast).
EVENTS = {
    '2018': {'pre': ('2018-04-01', '2018-05-25'), 'flood': ('2018-08-08', '2018-08-31'), 'pop_year': 2018},
    '2019': {'pre': ('2019-04-01', '2019-05-25'), 'flood': ('2019-08-05', '2019-08-25'), 'pop_year': 2019},
}

# Flood detection thresholds (change detection on VV backscatter, dB).
VV_THRESHOLD   = -15.0   # water is dark: VV during flood below this
DIFF_THRESHOLD = 3.0     # backscatter must drop by at least this much vs pre-flood
SLOPE_MAX_DEG  = 5       # floods do not stay on steep slopes
MIN_PATCH_PX   = 8       # remove isolated speckle pixels
# ==================================================================

In [ ]:
import ee, time, json
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print('Earth Engine ready')

## District boundaries
FAO GAUL level 2 is used. Check that 14 districts are printed. If the names or count are wrong, upload a Kerala district shapefile (e.g. from the Survey of India or DataMeet) as an Earth Engine asset and put its path in `CUSTOM_DISTRICT_ASSET`.

In [ ]:
CUSTOM_DISTRICT_ASSET = None   # e.g. 'projects/your-project/assets/kerala_districts'
NAME_FIELD = 'ADM2_NAME'

if CUSTOM_DISTRICT_ASSET:
    districts = ee.FeatureCollection(CUSTOM_DISTRICT_ASSET)
else:
    districts = (ee.FeatureCollection('FAO/GAUL/2015/level2')
                 .filter(ee.Filter.eq('ADM1_NAME', 'Kerala')))
districts = districts.map(lambda f: f.set('district', f.get(NAME_FIELD)))
region = districts.geometry()

names = districts.aggregate_array('district').getInfo()
print(len(names), 'districts:', sorted(names))

ee.batch.Export.table.toDrive(
    collection=districts.select(['district']), description='kerala_districts',
    folder=DRIVE_FOLDER, fileFormat='GeoJSON').start()

## A + B. Sentinel-1 flood extent and exposed population

Method (same idea as Tiwari et al. 2020, PLOS ONE, used in the paper):
1. Sentinel-1 GRD, IW mode, VV polarization; 50 m focal-median speckle filter.
2. Pre-flood reference = median of the dry-season images **from the same orbit pass** (ascending/descending) as the flood image.
3. Flood = VV below `VV_THRESHOLD` **and** a drop of more than `DIFF_THRESHOLD` dB against the reference.
4. Remove permanent water (JRC Global Surface Water, ≥10 months/year), steep slopes (SRTM) and small patches.
5. Exposed population = WorldPop 100 m population inside the flood mask, summed per district.

`imaged_km2` tells you how much of each district the satellite actually covered on that date. A district with little coverage on one date should take its flood state from the nearest covered date.

In [ ]:
def s1(start, end):
    return (ee.ImageCollection('COPERNICUS/S1_GRD')
            .filterBounds(region).filterDate(start, end)
            .filter(ee.Filter.eq('instrumentMode', 'IW'))
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
            .select('VV'))

def smooth(img):
    return img.focal_median(50, 'circle', 'meters')

gsw   = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('seasonality').unmask(0)
slope = ee.Terrain.slope(ee.Image('USGS/SRTMGL1_003'))
area  = ee.Image.pixelArea()

def flood_mask(date_str, pre_window):
    day = s1(date_str, ee.Date(date_str).advance(1, 'day'))
    orbit_pass = day.first().get('orbitProperties_pass')
    during = smooth(day.mosaic())
    before = smooth(s1(*pre_window).filter(ee.Filter.eq('orbitProperties_pass', orbit_pass)).median())
    flood = (during.lt(VV_THRESHOLD)
             .And(before.subtract(during).gt(DIFF_THRESHOLD))
             .updateMask(gsw.lt(10))
             .updateMask(slope.lt(SLOPE_MAX_DEG)))
    flood = flood.selfMask()
    flood = flood.updateMask(flood.connectedPixelCount(MIN_PATCH_PX + 1).gte(MIN_PATCH_PX))
    imaged = during.mask().gt(0)
    return flood.unmask(0).updateMask(imaged).rename('flood').toByte(), imaged

def population(year):
    pop = (ee.ImageCollection('WorldPop/GP/100m/pop')
           .filter(ee.Filter.eq('country', 'IND'))
           .filter(ee.Filter.eq('year', year)).first())
    # people per m2 in WorldPop's own grid, so sums are correct at any scale
    return pop.divide(ee.Image.pixelArea().reproject(pop.projection()))

def tag(fc, d, ev):
    return fc.map(lambda x: x.set({'date': d, 'event': ev}))

tasks = []
for ev, cfg in EVENTS.items():
    dates = (s1(*cfg['flood']).aggregate_array('system:time_start')
             .map(lambda t: ee.Date(t).format('YYYY-MM-dd')).distinct().sort().getInfo())
    print(f'{ev}: Sentinel-1 dates in flood window ->', dates)
    dens = population(cfg['pop_year'])
    for d in dates:
        flood, imaged = flood_mask(d, cfg['pre'])
        f = flood.unmask(0)
        stats_img = ee.Image.cat([
            f.multiply(area).divide(1e6).rename('flood_km2'),
            imaged.unmask(0).multiply(area).divide(1e6).rename('imaged_km2'),
            dens.multiply(f).multiply(area).rename('exposed_pop'),
            dens.multiply(area).rename('total_pop'),
        ])
        stats = stats_img.reduceRegions(collection=districts.select(['district']),
                                        reducer=ee.Reducer.sum(), scale=20, tileScale=8)
        stats = tag(stats, d, ev)
        tag = d.replace('-', '')
        t1 = ee.batch.Export.table.toDrive(
            collection=stats, description=f'exposure_{tag}', folder=DRIVE_FOLDER,
            fileFormat='CSV',
            selectors=['event', 'date', 'district', 'flood_km2', 'imaged_km2', 'exposed_pop', 'total_pop'])
        t2 = ee.batch.Export.image.toDrive(
            image=flood, description=f'floodmask_{tag}', folder=DRIVE_FOLDER,
            region=region, scale=30, crs='EPSG:4326', maxPixels=1e10)
        t1.start(); t2.start(); tasks += [t1, t2]
print(len(tasks), 'export tasks started')

### Visual check (do this before trusting the numbers)
Red = detected flood on the chosen date. Compare with the published Kerala 2018 maps (e.g. Tiwari et al. 2020). If rivers and paddy areas such as Kuttanad (Alappuzha) are not flooded, lower `DIFF_THRESHOLD` to 2; if hills show flood, lower `SLOPE_MAX_DEG`.

In [ ]:
import geemap
CHECK_DATE = None   # e.g. '2018-08-21'; None = first 2018 date
ev = '2018'
if CHECK_DATE is None:
    CHECK_DATE = (s1(*EVENTS[ev]['flood']).aggregate_array('system:time_start')
                  .map(lambda t: ee.Date(t).format('YYYY-MM-dd')).distinct().sort().get(0).getInfo())
fm, _ = flood_mask(CHECK_DATE, EVENTS[ev]['pre'])
m = geemap.Map(center=[10.0, 76.4], zoom=8)
m.addLayer(districts.style(color='black', fillColor='00000000'), {}, 'districts')
m.addLayer(fm.selfMask(), {'palette': ['red']}, f'flood {CHECK_DATE}')
m

In [ ]:
# Wait for Earth Engine exports to finish
while True:
    states = [t.status()['state'] for t in tasks]
    done = sum(s in ('COMPLETED', 'FAILED', 'CANCELLED') for s in states)
    print(time.strftime('%H:%M:%S'), f'{done}/{len(tasks)} finished')
    if done == len(tasks):
        break
    time.sleep(60)
failed = [t.status() for t in tasks if t.status()['state'] != 'COMPLETED']
print('FAILED:', failed if failed else 'none')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, os, pandas as pd
OUT = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
csvs = sorted(glob.glob(f'{OUT}/exposure_*.csv'))
exp = pd.concat([pd.read_csv(c) for c in csvs], ignore_index=True)
exp['imaged_frac'] = exp['imaged_km2'] / exp.groupby('district')['imaged_km2'].transform('max')
exp = exp.sort_values(['event', 'date', 'district'])
exp.to_csv(f'{OUT}/kerala_flood_exposure_all.csv', index=False)
print(exp.groupby(['event', 'date'])[['flood_km2', 'exposed_pop']].sum().round(0))
exp.head(20)

## C. OpenStreetMap road network

Drivable main roads (motorway to tertiary) for all of Kerala, with speeds and travel times. Smaller roads are left out to keep the graph manageable; add `unclassified|residential` to `ROAD_FILTER` if you need village access.

In [ ]:
import osmnx as ox, networkx as nx, geopandas as gpd, numpy as np
ROAD_FILTER = '["highway"~"motorway|motorway_link|trunk|trunk_link|primary|primary_link|secondary|secondary_link|tertiary|tertiary_link"]'
G = ox.graph_from_place('Kerala, India', custom_filter=ROAD_FILTER, simplify=True)
G = ox.add_edge_speeds(G)        # fills missing speeds from road type
G = ox.add_edge_travel_times(G)  # seconds
ox.save_graphml(G, f'{OUT}/kerala_roads.graphml')
nodes, edges = ox.graph_to_gdfs(G)
edges = edges.reset_index()
edges['edge_id'] = np.arange(len(edges))
edges[['edge_id', 'u', 'v', 'key', 'highway', 'length', 'speed_kph', 'travel_time', 'geometry']].astype({'highway': str}).to_file(f'{OUT}/kerala_roads.gpkg', driver='GPKG')
print(len(nodes), 'nodes,', len(edges), 'edges')

### Normal travel times between districts
From each district's population-weighted centre (approximated here by a point inside the district) to every other. When you have the NDRF / state staging bases, add them to `BASES` and the base-to-district matrix $\tau^0_{j,i}$ is produced too.

In [ ]:
dist = gpd.read_file(f'{OUT}/kerala_districts.geojson').to_crs(4326)
dist['pt'] = dist.geometry.representative_point()
dist['node'] = ox.distance.nearest_nodes(G, dist.pt.x, dist.pt.y)

# Staging bases: name -> (lat, lon). Fill from NDRF / state deployment records.
BASES = {
    # 'Base name': (lat, lon),
}
origins = {r.district: r.node for r in dist.itertuples()}
for b, (lat, lon) in BASES.items():
    origins[b] = ox.distance.nearest_nodes(G, lon, lat)

Gu = G.to_undirected()  # avoid one-way artefacts between far-apart points
rows = []
for o_name, o_node in origins.items():
    tt = nx.single_source_dijkstra_path_length(Gu, o_node, weight='travel_time')
    for r in dist.itertuples():
        rows.append({'origin': o_name, 'district': r.district,
                     'tau0_hours': tt.get(r.node, np.nan) / 3600})
tau0 = pd.DataFrame(rows)
tau0.to_csv(f'{OUT}/kerala_travel_time_tau0.csv', index=False)
tau0.pivot(index='origin', columns='district', values='tau0_hours').round(2)

### Flooded road segments per Sentinel-1 date
A segment is marked flooded if any point sampled every 100 m along it falls in the flood mask. Landslide closures are added later from the landslide inventory.

In [ ]:
import rasterio
edges_m = edges.to_crs(32643)          # UTM 43N, metres
eid_list, geom_list = [], []
for eid, line in zip(edges_m.edge_id, edges_m.geometry):
    n = max(int(line.length // 100), 1)
    for i in range(n + 1):
        eid_list.append(eid)
        geom_list.append(line.interpolate(i / n, normalized=True))
pts = gpd.GeoSeries(geom_list, crs=32643).to_crs(4326)
coords = list(zip(pts.x, pts.y))
pt_edge = np.array(eid_list)
print(len(coords), 'sample points on', len(edges), 'segments')

records = []
for tif in sorted(glob.glob(f'{OUT}/floodmask_*.tif')):
    tag8 = os.path.basename(tif).split('_')[1][:8]
    date = f'{tag8[:4]}-{tag8[4:6]}-{tag8[6:]}'
    with rasterio.open(tif) as src:
        vals = np.array([v[0] for v in src.sample(coords)])
    for eid in np.unique(pt_edge[vals == 1]):
        records.append({'date': date, 'edge_id': int(eid)})
    print(date, 'done')
flooded = pd.DataFrame(records, columns=['date', 'edge_id'])
flooded.to_csv(f'{OUT}/kerala_flooded_roads.csv', index=False)
print(flooded.groupby('date').size().rename('flooded segments'))

## Outputs in `My Drive/Paper4_Kerala/`

| File | Contents |
|---|---|
| `kerala_districts.geojson` | district boundaries |
| `floodmask_YYYYMMDD.tif` | 30 m flood mask per Sentinel-1 date (1 = flood, 0 = dry, nodata = not imaged) |
| `kerala_flood_exposure_all.csv` | event, date, district, flood_km2, imaged_km2, exposed_pop, total_pop |
| `kerala_roads.graphml`, `kerala_roads.gpkg` | road network with travel times |
| `kerala_travel_time_tau0.csv` | normal travel time (h) from each district/base to each district |
| `kerala_flooded_roads.csv` | road segments flooded on each date |

**Write down for Section 5.3 of the paper:** the Sentinel-1 dates printed above, the thresholds used, and the total flooded area for 2018 (to compare with Tiwari et al. 2020).